阶段1：构造模型

在票价确定的前提下，预测各航段的客流量（已完成）。

阶段2：解析过程

根据客流信息分配舱位，以确定总收益。
客舱分为两类：短途类（AB或BC，称为S类）和长途类（AC，称为L类），总容量为C。

根据是否满座进行分类：

若不满座（S+L≤C），则按S、L和C的预测比例分配舱位，以实现收益最大化。
若满座（S+L>C），则需根据销售量变化预测进行取舍，并作如下判断：
若票价AB+BC>AC，则优先将舱位拆分为AB和BC段销售，S的值取min(AB, BC)，其余舱位分配给L。
若票价AB+BC<AC，则优先分配舱位给AC段，剩余舱位分配给AB和BC段。

阶段3：求总收益最大值

以票价为自变量，总收益为因变量，绘制总收益曲线，并求出总收益的最大值。

In [1]:
! which python

/opt/data/envs/machine/bin/python


## 加载CSV数据

### 加载CSV文件

In [2]:
import pandas as pd
import numpy as np

# 读取CSV文件
# /Users/zhanyu/code/data-hh/海航系销售结果数据
# df = pd.read_csv('../train_data/海航系销售结果数据.csv')
df = pd.read_csv('../train_data/海航系销售结果数据.csv', index_col=0)

# 显示前两行数据以确保正确加载
print(df.shape)
print(df.head(5))

(6546417, 16)
           日期      航段   航班号         航线    a    b    c 宽窄体      起飞时间   布局   机型  \
0  2023-01-01  TFULXA  111T  TFUGZGLXA  TFU  GZG  LXA  未知  18:00:00    0  NaN   
1  2023-01-01  GZGLXA  111T  TFUGZGLXA  TFU  GZG  LXA  未知  20:00:00    0  NaN   
2  2023-01-01  TFUGZG  111T  TFUGZGLXA  TFU  GZG  LXA  未知  18:00:00    0  NaN   
3  2023-01-01  LXAMIG  3017     LXAMIG  LXA  MIG  NaN  窄体  13:25:00  132  319   
4  2023-01-01  MIGLXA  3018     MIGLXA  MIG  LXA  NaN  窄体  16:15:00  132  319   

   航节数  航节号    航距      客票收入   客流  
0    3    3  0.00       0.0    0  
1    3    2  3.00       0.0    0  
2    3    1  1.00       0.0    0  
3    1    1  1.70  223630.0  127  
4    1    1  2.28   76854.0   60  


In [3]:
# 获取第0列的列名
first_column_name = df.columns[2]
print("第0列的列名:", first_column_name)

# 查看第0列的唯一值类型
unique_types = df.iloc[:, 2].apply(type).unique()
print("\n第0列中的数据类型：")
print(unique_types)

# 查看第0列的唯一值
unique_values = df.iloc[:, 2].unique()
print("\n第0列中的唯一值示例：")
print(unique_values[:-1])  # 显示前10个唯一值

# 统计每种类型的数量
type_counts = df.iloc[:, 2].apply(type).value_counts()
print("\n每种类型的数量：")
print(type_counts)

第0列的列名: 航班号

第0列中的数据类型：
[<class 'str'>]

第0列中的唯一值示例：
['111T' '3017' '3018' ... '981X' '631U' '974Y']

每种类型的数量：
航班号
<class 'str'>    6546417
Name: count, dtype: int64


### 修改数据字段名

In [4]:
import pandas as pd

# 假设 df 是你的原始 DataFrame

# 创建原列名与新列名的映射字典
column_mapping = {
    '日期': 'flt_date',
    '航段': 'segment',
    '航班号': 'flt_no',
    '宽窄体': 'bd_type',
    '起飞时间': 'dep_time',
    '布局': 'cap',
    '机型': 'aircraft',
    '航节数': 'legs',
    '航节号': 'leg_no',
    '航距': 'duration',
    '客票收入': 'tkt_rev',
    '客流': 'pax'
}

# 使用 rename 方法修改列名
df.rename(columns=column_mapping, inplace=True)

# 查看修改后的 DataFrame
print(df.head())

     flt_date segment flt_no         航线    a    b    c bd_type  dep_time  cap  \
0  2023-01-01  TFULXA   111T  TFUGZGLXA  TFU  GZG  LXA      未知  18:00:00    0   
1  2023-01-01  GZGLXA   111T  TFUGZGLXA  TFU  GZG  LXA      未知  20:00:00    0   
2  2023-01-01  TFUGZG   111T  TFUGZGLXA  TFU  GZG  LXA      未知  18:00:00    0   
3  2023-01-01  LXAMIG   3017     LXAMIG  LXA  MIG  NaN      窄体  13:25:00  132   
4  2023-01-01  MIGLXA   3018     MIGLXA  MIG  LXA  NaN      窄体  16:15:00  132   

  aircraft  legs  leg_no  duration   tkt_rev  pax  
0      NaN     3       3      0.00       0.0    0  
1      NaN     3       2      3.00       0.0    0  
2      NaN     3       1      1.00       0.0    0  
3      319     1       1      1.70  223630.0  127  
4      319     1       1      2.28   76854.0   60  


In [5]:
# 获取第0列的列名
first_column_name = df.columns[2]
print("第0列的列名:", first_column_name)

# 查看第0列的唯一值类型
unique_types = df.iloc[:, 2].apply(type).unique()
print("\n第0列中的数据类型：")
print(unique_types)

# 查看第0列的唯一值
unique_values = df.iloc[:, 2].unique()
print("\n第0列中的唯一值示例：")
print(unique_values[:-1])  # 显示前10个唯一值

# 统计每种类型的数量
type_counts = df.iloc[:, 2].apply(type).value_counts()
print("\n每种类型的数量：")
print(type_counts)

第0列的列名: flt_no

第0列中的数据类型：
[<class 'str'>]

第0列中的唯一值示例：
['111T' '3017' '3018' ... '981X' '631U' '974Y']

每种类型的数量：
flt_no
<class 'str'>    6546417
Name: count, dtype: int64


## 删除航线字段

In [6]:
# 删除 '航线' 列
df.drop(columns=['航线'], inplace=True)

# 查看删除后的 DataFrame
print(df.head())

     flt_date segment flt_no    a    b    c bd_type  dep_time  cap aircraft  \
0  2023-01-01  TFULXA   111T  TFU  GZG  LXA      未知  18:00:00    0      NaN   
1  2023-01-01  GZGLXA   111T  TFU  GZG  LXA      未知  20:00:00    0      NaN   
2  2023-01-01  TFUGZG   111T  TFU  GZG  LXA      未知  18:00:00    0      NaN   
3  2023-01-01  LXAMIG   3017  LXA  MIG  NaN      窄体  13:25:00  132      319   
4  2023-01-01  MIGLXA   3018  MIG  LXA  NaN      窄体  16:15:00  132      319   

   legs  leg_no  duration   tkt_rev  pax  
0     3       3      0.00       0.0    0  
1     3       2      3.00       0.0    0  
2     3       1      1.00       0.0    0  
3     1       1      1.70  223630.0  127  
4     1       1      2.28   76854.0   60  


## 拆分 flt_date 为 year, month, day, weekday

In [7]:
# 将 'flt_date' 列转换为 datetime 格式
df['flt_date'] = pd.to_datetime(df['flt_date'])

# 提取年、月、日和星期几
df['year'] = df['flt_date'].dt.year
df['month'] = df['flt_date'].dt.month
df['day'] = df['flt_date'].dt.day
df['weekday'] = df['flt_date'].dt.weekday  # 0 = Monday, 6 = Sunday

# 删除原始的 'flt_date' 字段
df = df.drop(columns=['flt_date'])

# 查看结果
print(df.head(5))

  segment flt_no    a    b    c bd_type  dep_time  cap aircraft  legs  leg_no  \
0  TFULXA   111T  TFU  GZG  LXA      未知  18:00:00    0      NaN     3       3   
1  GZGLXA   111T  TFU  GZG  LXA      未知  20:00:00    0      NaN     3       2   
2  TFUGZG   111T  TFU  GZG  LXA      未知  18:00:00    0      NaN     3       1   
3  LXAMIG   3017  LXA  MIG  NaN      窄体  13:25:00  132      319     1       1   
4  MIGLXA   3018  MIG  LXA  NaN      窄体  16:15:00  132      319     1       1   

   duration   tkt_rev  pax  year  month  day  weekday  
0      0.00       0.0    0  2023      1    1        6  
1      3.00       0.0    0  2023      1    1        6  
2      1.00       0.0    0  2023      1    1        6  
3      1.70  223630.0  127  2023      1    1        6  
4      2.28   76854.0   60  2023      1    1        6  


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6546417 entries, 0 to 6546416
Data columns (total 18 columns):
 #   Column    Dtype  
---  ------    -----  
 0   segment   object 
 1   flt_no    object 
 2   a         object 
 3   b         object 
 4   c         object 
 5   bd_type   object 
 6   dep_time  object 
 7   cap       int64  
 8   aircraft  object 
 9   legs      int64  
 10  leg_no    int64  
 11  duration  float64
 12  tkt_rev   float64
 13  pax       int64  
 14  year      int32  
 15  month     int32  
 16  day       int32  
 17  weekday   int32  
dtypes: float64(2), int32(4), int64(4), object(8)
memory usage: 849.1+ MB


## 拆分 dep_time 为 hour, minute, second

In [9]:
# 拆分 'dep_time' 字段
df[['hour', 'minute', 'second']] = df['dep_time'].str.split(':', expand=True)

# 将拆分的结果转换为整数
df['hour'] = df['hour'].astype(int)
df['minute'] = df['minute'].astype(int)
df['second'] = df['second'].astype(int)

# 删除原始的 'dep_time' 字段
df = df.drop(columns=['dep_time'])

# 查看结果
print(df.head(5))

  segment flt_no    a    b    c bd_type  cap aircraft  legs  leg_no  duration  \
0  TFULXA   111T  TFU  GZG  LXA      未知    0      NaN     3       3      0.00   
1  GZGLXA   111T  TFU  GZG  LXA      未知    0      NaN     3       2      3.00   
2  TFUGZG   111T  TFU  GZG  LXA      未知    0      NaN     3       1      1.00   
3  LXAMIG   3017  LXA  MIG  NaN      窄体  132      319     1       1      1.70   
4  MIGLXA   3018  MIG  LXA  NaN      窄体  132      319     1       1      2.28   

    tkt_rev  pax  year  month  day  weekday  hour  minute  second  
0       0.0    0  2023      1    1        6    18       0       0  
1       0.0    0  2023      1    1        6    20       0       0  
2       0.0    0  2023      1    1        6    18       0       0  
3  223630.0  127  2023      1    1        6    13      25       0  
4   76854.0   60  2023      1    1        6    16      15       0  


## 拆分 segment 为 from 和 to

In [10]:
# 计算 'segment' 列每个值的长度
segment_lengths = df['segment'].apply(len)

In [11]:
# 统计每个长度值出现的次数
length_counts = segment_lengths.value_counts()

In [12]:
# 获取 'segment_length' 列的统计描述
length_description = segment_lengths.describe()

# 打印统计描述
print(length_description)

count    6546417.0
mean           6.0
std            0.0
min            6.0
25%            6.0
50%            6.0
75%            6.0
max            6.0
Name: segment, dtype: float64


In [13]:
# 提取 'segment' 列的前三个字符作为 'from' 列
df['from'] = df['segment'].str[:3]

# 提取 'segment' 列的后三个字符作为 'to' 列
df['to'] = df['segment'].str[-3:]

# 删除原始的 'segment' 字段
df = df.drop(columns=['segment'])

# 查看结果
print(df.head(10))

  flt_no    a    b    c bd_type  cap aircraft  legs  leg_no  duration  ...  \
0   111T  TFU  GZG  LXA      未知    0      NaN     3       3      0.00  ...   
1   111T  TFU  GZG  LXA      未知    0      NaN     3       2      3.00  ...   
2   111T  TFU  GZG  LXA      未知    0      NaN     3       1      1.00  ...   
3   3017  LXA  MIG  NaN      窄体  132      319     1       1      1.70  ...   
4   3018  MIG  LXA  NaN      窄体  132      319     1       1      2.28  ...   
5   3051  CAN  LZO  NaN      窄体  164      320     1       1      1.63  ...   
6   3052  LZO  CAN  NaN      窄体  164      320     1       1      1.55  ...   
7   3053  CAN  YBP  NaN      窄体  194      321     1       1      1.70  ...   
8   3054  YBP  CAN  NaN      窄体  194      321     1       1      1.62  ...   
9   3055  CAN  WXN  NaN      窄体  164      320     1       1      1.55  ...   

   pax  year  month  day  weekday  hour  minute  second  from   to  
0    0  2023      1    1        6    18       0       0   TFU  LXA  
1  

## 删除pax异常的行

In [14]:
# 统计 'pax' 字段中缺失值的行数
missing_pax = df[df['pax'].isna()]

# 统计 'pax' 字段中0值的行数
zero_pax = df[df['pax'] == 0]

# 输出统计结果
num_missing = missing_pax.shape[0]
num_zero = zero_pax.shape[0]

print(df.shape[0])
print(f"缺失值的行数: {num_missing}")
print(f"为0的行数: {num_zero}")

6546417
缺失值的行数: 0
为0的行数: 166560


In [15]:
# 删除 'pax' 字段为缺失值或为0的行
df = df.dropna(subset=['pax'])  # 删除pax列中的缺失值行
df = df[df['pax'] != 0]  # 删除pax列中为0的行

# 查看删除后的DataFrame行数
num_rows_after_cleanup = df.shape[0]
print(f"删除缺失值或为0的行后，DataFrame一共有 {num_rows_after_cleanup} 行")

删除缺失值或为0的行后，DataFrame一共有 6379857 行


In [16]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def remove_outliers(data, col):
    """删除指定列中的异常值"""
    Q1 = data[col].quantile(0.25)  # 第1四分位数
    Q3 = data[col].quantile(0.75)  # 第3四分位数
    IQR = Q3 - Q1  # 四分位距
    lower_bound = Q1 - 1.5 * IQR  # 下限
    upper_bound = Q3 + 1.5 * IQR  # 上限
    
    # 筛选出范围内的数据
    filtered_data = data[(data[col] >= lower_bound) & (data[col] <= upper_bound)]
    return filtered_data

def plot(data, col):
    """绘制指定列的分布图和箱型图"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
    sns.histplot(data[col], kde=True, ax=ax1)  # 替换 distplot 为 histplot
    sns.boxplot(data[col], ax=ax2)
    plt.tight_layout()
    plt.show()

# # 绘制过滤后的数据分布图和箱型图
# plot(df, 'pax')
# # 示例数据框
# # 假设 df 是你的数据框
# # 删除 pax 列中的异常值
# # df = remove_outliers(df, 'pax')

# # 绘制过滤后的数据分布图和箱型图
# plot(df, 'pax')


In [17]:
# 删除 pax 小于 20 的行
# df = df[df['pax'] >= 20]

## 处理aircraft

In [18]:
# 查看 'cap' 列中每个元素的数据类型
print(df['aircraft'].apply(type).value_counts())

aircraft
<class 'str'>      6379751
<class 'float'>        106
Name: count, dtype: int64


In [19]:
# 统计 'pax' 字段中缺失值的行数
missing_pax = df[df['aircraft'].isna()]

# 统计 'pax' 字段中0值的行数
zero_pax = df[df['aircraft'] == 0]

# 输出统计结果
num_missing = missing_pax.shape[0]
num_zero = zero_pax.shape[0]

print(df.shape[0])
print(f"缺失值的行数: {num_missing}")
print(f"为0的行数: {num_zero}")

6379857
缺失值的行数: 106
为0的行数: 0


In [20]:
# 将 cap 列转换为 Pandas 的 Int64 类型，这种类型支持 NaN 值
# 将 'aircraft' 列转换为字符串类型
# 用 'Unknown' 填充 NaN 值
df['aircraft'] = df['aircraft'].fillna('Unknown')
df['aircraft'] = df['aircraft'].astype(str)

In [21]:
# 查看 'cap' 列中每个元素的数据类型
print(df['aircraft'].apply(type).value_counts())

aircraft
<class 'str'>    6379857
Name: count, dtype: int64


## 处理cap

In [22]:
# 查看 'cap' 列中每个元素的数据类型
print(df['cap'].apply(type).value_counts())

cap
<class 'int'>    6379857
Name: count, dtype: int64


In [23]:
# 统计 'pax' 字段中缺失值的行数
missing_pax = df[df['cap'].isna()]

# 统计 'pax' 字段中0值的行数
zero_pax = df[df['cap'] == 0]

# 输出统计结果
num_missing = missing_pax.shape[0]
num_zero = zero_pax.shape[0]

print(df.shape[0])
print(f"缺失值的行数: {num_missing}")
print(f"为0的行数: {num_zero}")

# 将 'cap' 字段中为0的值修改为 NaN
df.loc[df['cap'] == 0, 'cap'] = np.nan

# 将 'cap' 字段转换为字符串类型
df['cap'] = df['cap'].astype(str)



6379857
缺失值的行数: 0
为0的行数: 813340


In [24]:
# 查看 'cap' 列中每个元素的数据类型
print(df['cap'].apply(type).value_counts())
print(df['aircraft'].apply(type).value_counts())

cap
<class 'str'>    6379857
Name: count, dtype: int64
aircraft
<class 'str'>    6379857
Name: count, dtype: int64


## 处理duration（待做）

In [25]:
# 将 'duration' 列中值为 0 的部分替换为 NaN
df.loc[df['duration'] == 0, 'duration'] = np.nan

## 处理tkt_rev

有一些异常行的tkt_rev为0，pax却不为0

In [26]:
# 统计 'unit_price' 列中为 0 的行数
zero_count = (df['tkt_rev'] == 0).sum()

# 统计 'unit_price' 列中为 NaN 的行数
nan_count = df['tkt_rev'].isna().sum()

# 输出结果
print(f"tkt_rev 列中为 0 的行数: {zero_count}")
print(f"tkt_rev 列中为 NaN 的行数: {nan_count}")

tkt_rev 列中为 0 的行数: 1146
tkt_rev 列中为 NaN 的行数: 0


In [27]:
df = df[df['tkt_rev'] != 0]

# 计算单价 'unit_price'，即 tkt_rev 除以 pax
df['unit_price'] = df['tkt_rev'] / df['pax']

# 删除 'tkt_rev' 列
df = df.drop(columns=['tkt_rev'])

## 保存文件

In [28]:
# 查看前几行数据，确保加载成功
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 6378711 entries, 3 to 6546416
Data columns (total 21 columns):
 #   Column      Dtype  
---  ------      -----  
 0   flt_no      object 
 1   a           object 
 2   b           object 
 3   c           object 
 4   bd_type     object 
 5   cap         object 
 6   aircraft    object 
 7   legs        int64  
 8   leg_no      int64  
 9   duration    float64
 10  pax         int64  
 11  year        int32  
 12  month       int32  
 13  day         int32  
 14  weekday     int32  
 15  hour        int64  
 16  minute      int64  
 17  second      int64  
 18  from        object 
 19  to          object 
 20  unit_price  float64
dtypes: float64(2), int32(4), int64(6), object(9)
memory usage: 973.3+ MB
None


In [30]:
# 先获取2024年6月的数据
condition_2024_06 = (df['year'] == 2024) & (df['month'] == 6)
df_2024_06 = df[condition_2024_06]

# 其余数据为2023-2024.05的数据
df_2023_2024_05 = df[~condition_2024_06]

# 保存两个时间段的数据
df_2023_2024_05.to_csv('../my/hh_result/result_2023_202405_pax>0.csv', index=False)
df_2024_06.to_csv('../my/hh_result/result_202406_pax>0.csv', index=False)

# 打印两个数据集的基本信息
print("2023-2024.05数据集大小:", len(df_2023_2024_05))
print("2024.06数据集大小:", len(df_2024_06))

2023-2024.05数据集大小: 6025727
2024.06数据集大小: 352984
